In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from functools import reduce
import joblib
from typing import Dict
import gc

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.utils.data_structs import TripletCreator, NodeCreator, Relation, NODES_TYPES_MAP, RELATIONS_TYPES_MAP
from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

NEO4J_URL ="bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PWD = "password"

GRAPH_DB_NAME = 'diaasq3'
DATASET_PATH = '../data/Augment_DiaASQ.json'
LOAD_EXTRACTED_TRIPLETS_FILE = "../data/tmp_new_graph_extracted/tmp_extracted_openai_gpt4omini_triplets.json"
gc.collect()

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4

### Update

* config1: (testdb) - golden-граф
    * vectorized nodes = 8
    * vectorized triplets = 4 
* config2: (diaasq2) - build-грфа (c добавленным полем time)
    * vectorized nodes = 10
    * vectorized triplets = 6
* config3: (diaasq3) - build-граф (diaasq2 с добавленным полем name для realtions)
    * vectorized nodes = 11
    * vectorized triplets = 7  

In [2]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri=NEO4J_URL, user=NEO4J_USER, pwd=NEO4J_PWD, db_name=GRAPH_DB_NAME),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            '../data/graph_structures/vectorized_nodes/v11/densedb', 'vectorized_nodes', is_exist=True, need_to_clear=True
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../data/graph_structures/vectorized_triplets/v7/densedb', 'vectorized_triplets', is_exist=True, need_to_clear=True
        )
    ))
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [11]:
base_dialogs = json.loads(open(DATASET_PATH, 'r', encoding='utf-8').read())
raw_time = [dialogue['time'].split(', ')[0].strip() for dialogue in base_dialogs['data']]
print(len(raw_time))

3483


In [12]:
extracted_triplets = json.loads(open(LOAD_EXTRACTED_TRIPLETS_FILE, 'r', encoding='utf-8').read())
print(len(extracted_triplets))

3483


In [13]:
# adding time
not_str_counter = 0
all_items_counter = 0
for group_idx in tqdm(range(len(extracted_triplets))):
    cur_time = raw_time[group_idx]
    for triplet_idx in range(len(extracted_triplets[group_idx])):
        for item_idx in range(len(extracted_triplets[group_idx][triplet_idx])):
            if type(extracted_triplets[group_idx][triplet_idx][item_idx]['name']) is not str:
                not_str_counter += 1
            
            all_items_counter += 1
            extracted_triplets[group_idx][triplet_idx][item_idx]['name'] = str(extracted_triplets[group_idx][triplet_idx][item_idx]['name']).strip()

        if extracted_triplets[group_idx][triplet_idx][1]['prop']['type'] == 'simple':
            extracted_triplets[group_idx][triplet_idx][1]['prop']['time'] = cur_time
        else:
            extracted_triplets[group_idx][triplet_idx][2]['prop']['time'] = cur_time

print(not_str_counter, all_items_counter)

100%|██████████| 3483/3483 [00:00<00:00, 5115.77it/s]

1399 849804


In [14]:
extracted_triplets = reduce(lambda acc, v: acc + v, extracted_triplets, [])
print(len(extracted_triplets))

283268


In [15]:
formated_triplets = []
for raw_triplet in tqdm(extracted_triplets):
    formated_triplets.append(TripletCreator.create(
        NodeCreator.create(
            name=raw_triplet[0]['name'], type=NODES_TYPES_MAP[raw_triplet[0]['type']], 
            prop=raw_triplet[0]['prop'], add_stringified_node=False),
        Relation(name=raw_triplet[1]['name'], type=RELATIONS_TYPES_MAP[raw_triplet[1]['prop']['type']], 
                 prop={k: v for k, v in raw_triplet[1]['prop'].items() if k != 'type'}),
        NodeCreator.create(name=raw_triplet[2]['name'], type=NODES_TYPES_MAP[raw_triplet[2]['type']], 
                           prop=raw_triplet[2]['prop'], add_stringified_node=False)
    ))

100%|██████████| 283268/283268 [00:04<00:00, 63777.09it/s] 


In [16]:
kg_model.graph_db.execute_query("match (a) -[r] -> () delete a, r")
kg_model.graph_db.execute_query("match (a) delete a")

[]

In [17]:
kg_model.graph_db.create_triplets(formated_triplets)

  3%|▎         | 7311/283268 [11:13<10:47:53,  7.10it/s]

In [3]:
raw_triplets = list(kg_model.graph_db.execute_query("MATCH (n1)-[rel]->(n2) RETURN n1, rel, n2"))
formated_triplets = []
for raw_triplet in tqdm(raw_triplets):
    start_node = NodeCreator.create(id=raw_triplet['n1'].element_id, name=str(raw_triplet['n1']['name']), 
                                        type=NODES_TYPES_MAP[list(raw_triplet['n1'].labels)[0]],
                                        prop=dict(raw_triplet['n1']))
    end_node = NodeCreator.create(id=raw_triplet['n2'].element_id, name=str(raw_triplet['n2']['name']), 
                                        type=NODES_TYPES_MAP[list(raw_triplet['n2'].labels)[0]],
                                        prop=dict(raw_triplet['n2']))
    relation = Relation(id=raw_triplet['rel'].element_id, name=str(raw_triplet['rel']['name']), 
                            type=RELATIONS_TYPES_MAP[raw_triplet['rel'].type], 
                            prop=dict(raw_triplet['rel']))
    
    triplet = TripletCreator.create(start_node, relation, end_node, add_stringified_triplet=False)
    formated_triplets.append(triplet)

100%|██████████| 283268/283268 [00:08<00:00, 32690.70it/s]


In [ ]:
kg_model.embeddings_db.add_triplets(formated_triplets, batch_size=128)

all/unique_triplets - 283268/72280
all/unique_nodes - 566536/71338

In [5]:
kg_model.graph_db.close()